# 🐱🐶 Cat vs Dog Image Classifier using a CNN (TensorFlow/Keras)

**Run on GPU:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

CNN concepts covered: Convolution layers, Batch Normalization, ReLU activation, Max Pooling, Dropout, Global Average Pooling, Dense layers, L2 regularization, Data Augmentation, Adam optimizer, Binary cross-entropy loss, Early stopping, LR scheduling, Model checkpointing.

Graphs: sample images, class distribution, pixel histogram, accuracy/loss curves, confusion matrix, ROC curve, predictions grid, feature maps.

## 1. Setup and GPU check

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, auc

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", gpus if gpus else "NO GPU -> Runtime > Change runtime type > T4 GPU")

IMG_SIZE = 128
BATCH = 64
EPOCHS = 20
keras.utils.set_random_seed(42)

## 2. Load the dataset (Microsoft Cats vs Dogs, ~23k images)
Split: 80% train, 10% validation, 10% test.

In [ ]:
(raw_train, raw_val, raw_test), info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    as_supervised=True, with_info=True)

class_names = info.features['label'].names   # ['cat', 'dog']
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE

def prep(img, label):
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(img, tf.uint8), label      # uint8 keeps the cache small

train_c = raw_train.map(prep, AUTOTUNE).cache()
val_c   = raw_val.map(prep, AUTOTUNE).cache()
test_c  = raw_test.map(prep, AUTOTUNE).cache()

train_ds = train_c.shuffle(2000).batch(BATCH).prefetch(AUTOTUNE)
val_ds   = val_c.batch(BATCH).prefetch(AUTOTUNE)
test_ds  = test_c.batch(BATCH).prefetch(AUTOTUNE)

## 3. Explore the data (graphs)

In [ ]:
# Collect labels (this also fills the cache, so later epochs are fast)
train_labels = np.concatenate([y.numpy() for _, y in train_c.batch(1024)])
val_labels   = np.concatenate([y.numpy() for _, y in val_c.batch(1024)])
test_labels  = np.concatenate([y.numpy() for _, y in test_c.batch(1024)])

# Graph 1: class distribution in each split
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5), sharey=False)
for a, (name, lab) in zip(ax, [('Train', train_labels), ('Validation', val_labels), ('Test', test_labels)]):
    counts = np.bincount(lab, minlength=2)
    a.bar(class_names, counts, color=['#4C78A8', '#F58518'])
    a.set_title(f'{name} ({len(lab)} images)')
    for i, c in enumerate(counts):
        a.text(i, c, str(c), ha='center', va='bottom')
plt.suptitle('Class distribution'); plt.tight_layout(); plt.show()

# Graph 2: sample images
images, labels = next(iter(train_c.batch(16)))
plt.figure(figsize=(10, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].numpy()); plt.title(class_names[labels[i]]); plt.axis('off')
plt.suptitle('Sample training images'); plt.tight_layout(); plt.show()

# Graph 3: pixel intensity histogram per class
imgs, labs = next(iter(train_c.batch(512)))
imgs, labs = imgs.numpy(), labs.numpy()
plt.figure(figsize=(7, 3.5))
for idx, color in zip(range(2), ['#4C78A8', '#F58518']):
    plt.hist(imgs[labs == idx].ravel(), bins=50, alpha=0.5, color=color, label=class_names[idx], density=True)
plt.xlabel('Pixel value (0-255)'); plt.ylabel('Density'); plt.title('Pixel intensity distribution'); plt.legend(); plt.show()

## 4. Build the CNN
Four convolution blocks (32 → 64 → 128 → 256 filters). Each block = Conv → BatchNorm → ReLU (twice) → MaxPooling → Dropout.
Data augmentation layers only act during training.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='augmentation')

def conv_block(x, filters, idx, dropout):
    for j in (1, 2):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f'conv{idx}_{j}')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(dropout)(x)
    return x

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)                 # normalize pixels to 0-1

x = conv_block(x, 32, 1, 0.1)
x = conv_block(x, 64, 2, 0.2)
x = conv_block(x, 128, 3, 0.3)
x = conv_block(x, 256, 4, 0.3)

x = layers.GlobalAveragePooling2D()(x)          # replaces Flatten, fewer parameters
x = layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)   # probability of "dog"

model = keras.Model(inputs, outputs, name='cat_dog_cnn')
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()

## 5. Train on the GPU

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
    keras.callbacks.ModelCheckpoint('best_cat_dog_cnn.keras', monitor='val_accuracy', save_best_only=True),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

## 6. Training graphs

In [ ]:
hist = history.history
epochs_range = range(1, len(hist['loss']) + 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs_range, hist['accuracy'], 'o-', label='Train')
ax[0].plot(epochs_range, hist['val_accuracy'], 'o-', label='Validation')
ax[0].set_title('Accuracy'); ax[0].set_xlabel('Epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(epochs_range, hist['loss'], 'o-', label='Train')
ax[1].plot(epochs_range, hist['val_loss'], 'o-', label='Validation')
ax[1].set_title('Loss'); ax[1].set_xlabel('Epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}")

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=class_names))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax[0], cmap='Blues', colorbar=False)
ax[0].set_title('Confusion matrix')

# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
ax[1].plot(fpr, tpr, label=f'AUC = {auc(fpr, tpr):.3f}')
ax[1].plot([0, 1], [0, 1], 'k--')
ax[1].set_xlabel('False positive rate'); ax[1].set_ylabel('True positive rate')
ax[1].set_title('ROC curve'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Predictions on test images (green = correct, red = wrong)

In [ ]:
imgs, labs = next(iter(test_ds.unbatch().batch(16)))
probs = model.predict(imgs, verbose=0).ravel()

plt.figure(figsize=(11, 7))
for i in range(12):
    pred = int(probs[i] > 0.5)
    conf = probs[i] if pred == 1 else 1 - probs[i]
    plt.subplot(3, 4, i + 1)
    plt.imshow(imgs[i].numpy())
    plt.title(f"{class_names[pred]} ({conf:.0%})", color='green' if pred == labs[i] else 'red')
    plt.axis('off')
plt.tight_layout(); plt.show()

## 9. Visualize what the CNN sees (feature maps)
Early layers detect edges and colours; deeper layers detect more abstract shapes.

In [ ]:
layer_names = ['conv1_2', 'conv2_2', 'conv3_2', 'conv4_2']
extractor = keras.Model(model.input, [model.get_layer(n).output for n in layer_names])

sample = imgs[0:1].numpy().astype('float32')
feature_maps = extractor.predict(sample, verbose=0)

plt.figure(figsize=(3, 3)); plt.imshow(sample[0].astype('uint8')); plt.title('Input'); plt.axis('off'); plt.show()

for name, fm in zip(layer_names, feature_maps):
    n = 8
    plt.figure(figsize=(14, 2))
    for i in range(n):
        plt.subplot(1, n, i + 1)
        plt.imshow(fm[0, :, :, i], cmap='viridis'); plt.axis('off')
    plt.suptitle(f'{name} - first {n} of {fm.shape[-1]} feature maps (size {fm.shape[1]}x{fm.shape[2]})')
    plt.show()

## 10. Try your own image (optional)

In [ ]:
from google.colab import files
uploaded = files.upload()

for fname in uploaded:
    img = keras.utils.load_img(fname, target_size=(IMG_SIZE, IMG_SIZE))
    arr = np.expand_dims(keras.utils.img_to_array(img), 0)
    p = model.predict(arr, verbose=0)[0][0]
    label = class_names[int(p > 0.5)]
    plt.imshow(img); plt.axis('off')
    plt.title(f"{label} ({max(p, 1 - p):.0%} confident)"); plt.show()

## 11. Save the model
Download `cat_dog_cnn.keras` from the Colab file panel (left side).

In [ ]:
model.save('cat_dog_cnn.keras')
print("Saved: cat_dog_cnn.keras")